
# gSASRec: demonstrating the overconfidence correction (Section 6.4)

This notebook is the companion to **Section 6.4**. It trains the same SASRec architecture
from Section 6.3 with three different settings of the gBCE calibration parameter `t`:

- **t = 0** — plain BCE, no correction. This is the overconfident baseline Section 6.4 diagnoses.
- **t = 0.75** — the chapter's default partial correction.
- **t = 1** — full calibration (β = α exactly).

For each, we measure both ranking quality (NDCG@10, HR@10) and **calibration**: how close
the model's predicted probability for the true next item is to a sane value, rather than
being driven toward 1.0 regardless of evidence.

Uses the same data pipeline as the main Chapter 6 notebook. Set `QUICK_TEST = True` to
verify everything runs in a few minutes before committing to the full MovieLens 25M run.


In [ ]:

QUICK_TEST = True

import os
import zipfile
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



## 1. Data (same pipeline as the Chapter 6 notebook)


In [ ]:

DATA_DIR = "ml-25m"
ZIP_PATH = "ml-25m.zip"
URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"

def download_movielens():
    if os.path.exists(DATA_DIR):
        print(f"{DATA_DIR} already present, skipping download.")
        return
    import requests
    print("Downloading MovieLens 25M (this is ~250MB)...")
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Done.")

download_movielens()


In [ ]:

ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

if QUICK_TEST:
    sample_users = ratings["userId"].drop_duplicates().sample(n=3000, random_state=SEED)
    ratings = ratings[ratings["userId"].isin(sample_users)].reset_index(drop=True)

item_ids = ratings["movieId"].unique()
item_id_map = {old: new for new, old in enumerate(sorted(item_ids), start=1)}
ratings["item_idx"] = ratings["movieId"].map(item_id_map)
num_items = len(item_id_map)

user_sequences = (
    ratings.groupby("userId")["item_idx"]
    .apply(list)
    .reset_index(drop=True)
)
user_sequences = user_sequences[user_sequences.apply(len) >= 3].reset_index(drop=True)

print(f"Interactions: {len(ratings):,}   Users: {len(user_sequences):,}   Items: {num_items:,}")


In [ ]:

MAX_LEN = 200

def truncate_and_pad(seq, max_len=MAX_LEN):
    seq = seq[-max_len:]
    pad_len = max_len - len(seq)
    return [0] * pad_len + seq

def build_split(sequences):
    train_seqs, test_inputs, test_targets = [], [], []
    for seq in sequences:
        train_seqs.append(seq[:-2])
        test_inputs.append(seq[:-1])
        test_targets.append(seq[-1])
    return train_seqs, test_inputs, test_targets

train_seqs, test_inputs, test_targets = build_split(user_sequences.tolist())

class SASRecTrainDataset(Dataset):
    def __init__(self, sequences, num_items, max_len=MAX_LEN, n_neg=256):
        self.sequences = [s for s in sequences if len(s) >= 2]
        self.num_items = num_items
        self.max_len = max_len
        self.n_neg = n_neg

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        input_seq = torch.tensor(truncate_and_pad(seq[:-1], self.max_len), dtype=torch.long)
        target_seq = torch.tensor(truncate_and_pad(seq[1:], self.max_len), dtype=torch.long)
        neg_items = torch.randint(1, self.num_items + 1, (self.max_len, self.n_neg))
        return input_seq, target_seq, neg_items

class SASRecEvalDataset(Dataset):
    def __init__(self, input_seqs, targets, max_len=MAX_LEN):
        self.input_seqs = input_seqs
        self.targets = targets
        self.max_len = max_len

    def __len__(self):
        return len(self.input_seqs)

    def __getitem__(self, idx):
        seq = truncate_and_pad(self.input_seqs[idx], self.max_len)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

test_dataset = SASRecEvalDataset(test_inputs, test_targets, max_len=MAX_LEN)



## 2. The model (Listing 6.1, unchanged)


In [ ]:

class SASRec(nn.Module):
    def __init__(self, num_items, max_len=200, hidden_dim=64,
                 num_layers=2, num_heads=2, dropout=0.2):
        super().__init__()
        self.num_items = num_items
        self.max_len = max_len
        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)
        self.input_dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(self, sequences):
        batch_size, seq_len = sequences.shape
        positions = torch.arange(seq_len, device=sequences.device)
        positions = positions.unsqueeze(0).expand(batch_size, -1)
        x = self.item_emb(sequences) + self.pos_emb(positions)
        x = self.input_dropout(x)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=sequences.device, dtype=torch.bool),
            diagonal=1,
        )
        padding_mask = (sequences == 0)
        # A left-padded position has zero valid keys under causal+padding masking
        # combined, which makes softmax produce NaN there -- and across multiple stacked
        # layers that NaN leaks into every position's output (0 attention weight times a
        # NaN value is still NaN, not zero). Loop layers manually and sanitize between them.
        for layer in self.transformer.layers:
            x = layer(x, src_mask=causal_mask, src_key_padding_mask=padding_mask)
            x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)
        return self.final_norm(x)



## 3. The derivation from Section 6.4, made concrete

Before training anything, let's compute α and β for this exact dataset and negative-sampling
setup, the way Section 6.4 works through the MovieLens 25M / 256-negatives example.


In [ ]:

N_NEG = 256 if not QUICK_TEST else 64

def alpha_beta(num_items, n_neg, t):
    alpha = n_neg / max(num_items - 1, 1)
    beta = alpha * (t * (1 - 1 / alpha) + 1 / alpha)
    return alpha, beta

alpha, _ = alpha_beta(num_items, N_NEG, t=0)
print(f"Catalog size: {num_items:,}   Negatives per positive: {N_NEG}")
print(f"Sampling rate alpha = {alpha:.6f}\n")

print(f"{'t':>6}{'beta':>10}   meaning")
print("-" * 50)
for t in [0.0, 0.5, 0.75, 1.0]:
    _, beta = alpha_beta(num_items, N_NEG, t)
    note = ""
    if t == 0.0:
        note = "= 1.0, plain BCE, no correction"
    elif t == 1.0:
        note = "= alpha, fully calibrated"
    print(f"{t:>6.2f}{beta:>10.4f}   {note}")



## 4. gBCE loss (Listing 6.2) and the training loop


In [ ]:

def gbce_loss(pos_scores, neg_scores, pos_mask, num_items, t=0.75):
    n_neg = neg_scores.shape[-1]
    alpha = n_neg / max(num_items - 1, 1)
    beta = alpha * (t * (1.0 - 1.0 / alpha) + 1.0 / alpha)
    pos_loss = beta * F.softplus(-pos_scores)
    neg_loss = F.softplus(neg_scores).sum(dim=-1)
    per_position = (pos_loss + neg_loss) / (n_neg + 1)
    # softplus(-0) = log(2), not zero -- padding must be excluded after computing the
    # loss (not before, via masked_fill on the scores), and averaged over real positions only.
    per_position = per_position * pos_mask.float()
    return per_position.sum() / pos_mask.float().sum().clamp(min=1.0)

def train_sasrec(model, train_dataset, num_epochs, t, batch_size=128, lr=1e-3):
    model.to(device)
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        t0 = time.time()
        for input_seq, target_seq, neg_items in loader:
            input_seq, target_seq, neg_items = (
                input_seq.to(device), target_seq.to(device), neg_items.to(device)
            )
            hidden = model(input_seq)
            pos_mask = (target_seq != 0)
            pos_item_emb = model.item_emb(target_seq)
            pos_scores = (hidden * pos_item_emb).sum(-1)
            neg_item_emb = model.item_emb(neg_items)
            neg_scores = (hidden.unsqueeze(2) * neg_item_emb).sum(-1)

            loss = gbce_loss(pos_scores, neg_scores, pos_mask, model.num_items, t=t)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        print(f"  epoch {epoch+1}/{num_epochs}: loss={total_loss/n_batches:.4f}  ({time.time()-t0:.1f}s)")
    return model



## 5. Train one model per value of `t`

Same architecture, same data, same number of epochs — the only thing that changes between
runs is the calibration parameter.


In [ ]:

N_EPOCHS = 2 if QUICK_TEST else 20
HIDDEN_DIM = 64
T_VALUES = [0.0, 0.75, 1.0]

train_dataset = SASRecTrainDataset(train_seqs, num_items, max_len=MAX_LEN, n_neg=N_NEG)

models = {}
for t in T_VALUES:
    print(f"\nTraining with t={t} ...")
    torch.manual_seed(SEED)  # same init for every run, so differences come from t alone
    model = SASRec(num_items=num_items, max_len=MAX_LEN, hidden_dim=HIDDEN_DIM,
                    num_layers=2, num_heads=2, dropout=0.2)
    model = train_sasrec(model, train_dataset, num_epochs=N_EPOCHS, t=t, batch_size=128)
    models[t] = model



## 6. Measuring overconfidence and ranking quality side by side

Two numbers per model:

- **NDCG@10 / HR@10** — the usual ranking quality metrics.
- **Mean predicted probability on true positives** — `sigmoid(score)` for the held-out test
  target, averaged across users. This is the calibration signal: if it sits very close to
  1.0 regardless of `t`, the model has learned to be confident about *everything*, which is
  exactly the failure mode Section 6.4 describes. A well-calibrated model's average should
  be well below 1.0 — there is genuine uncertainty in next-item prediction, and a model that
  reports near-certainty on every prediction is miscalibrated, not skillful.


In [ ]:

def evaluate_with_calibration(model, eval_dataset, k=10, batch_size=256):
    model.eval()
    loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
    ndcgs, hits, pos_probs = [], [], []
    with torch.no_grad():
        for seqs, targets in loader:
            seqs, targets = seqs.to(device), targets.to(device)
            hidden = model(seqs)
            user_state = hidden[:, -1, :]
            scores = user_state @ model.item_emb.weight.T
            scores[:, 0] = -float("inf")

            topk = torch.topk(scores, k, dim=1).indices
            hit = (topk == targets.unsqueeze(1))
            rank = hit.float().argmax(dim=1)
            has_hit = hit.any(dim=1)
            ndcg = torch.where(has_hit, 1.0 / torch.log2(rank.float() + 2),
                                torch.zeros_like(rank, dtype=torch.float))
            ndcgs.extend(ndcg.cpu().tolist())
            hits.extend(has_hit.float().cpu().tolist())

            target_emb = model.item_emb(targets)
            target_score = (user_state * target_emb).sum(-1)
            pos_probs.extend(torch.sigmoid(target_score).cpu().tolist())
    return {
        "NDCG@10": float(np.mean(ndcgs)),
        "HR@10": float(np.mean(hits)),
        "MeanPosProb": float(np.mean(pos_probs)),
    }

print(f"{'t':>6}{'NDCG@10':>10}{'HR@10':>10}{'MeanPosProb':>14}")
print("-" * 42)
results = {}
for t, model in models.items():
    metrics = evaluate_with_calibration(model, test_dataset)
    results[t] = metrics
    print(f"{t:>6.2f}{metrics['NDCG@10']:>10.4f}{metrics['HR@10']:>10.4f}{metrics['MeanPosProb']:>14.4f}")



## 7. Reading the results

Look at the `MeanPosProb` column the way Section 6.4 frames it: **t = 0** (plain BCE) should
produce the highest mean positive probability of the three — the overconfidence the section
diagnoses, where the model has learned to say "yes, certainly" about almost every true
positive regardless of how genuinely predictable it was. As `t` increases toward 1, that
number should come down, trading some apparent confidence for a more honest estimate.

Whether NDCG@10 moves much across the three settings is itself informative. Petrov and
Macdonald's finding was that the *ranking* often survives reasonably well even when the
*probabilities* are badly overconfident — which is exactly why the problem was missed for
years: people were only checking ranking metrics. If your own run shows NDCG holding
roughly steady while MeanPosProb drops sharply from t=0 to t=1, that's the gSASRec paper's
core claim, reproduced on your own data: the architecture was fine all along; the
miscalibration was hiding in a place ranking metrics don't look.

**To reproduce the full chapter setup:** set `QUICK_TEST = False` and rerun — this uses the
same MovieLens 25M / 256-negatives / 20-epoch setup as the main Chapter 6 notebook, so the
NDCG@10 at t=0.75 here should land close to the number reported in Section 6.3.2.
